# Notebook 05.1: Full Fine-Tune — EfficientNet-B0 + DenseNet121 Ensemble

Tuned recipe aimed at closing the gap to the classmate's 0.94-on-held-out-test result (same architecture, 6 epochs). Changes vs. 05:

1. **Full fine-tune** — every pretrained parameter is unfrozen.
2. **Stronger augmentation** — bigger rotation / translation / jitter + `RandomErasing`. Chest-X-ray-safe ops only (no `RandAugment` Solarize/Posterize — those distort the radiographic signal).
3. **Single aggressive LR** — `1e-4` for both backbone and head. Required to actually move the backbone in a 6-epoch budget.
4. **`pos_weight` in BCE** — small but free correction for the slight class imbalance (pos rate 0.456).
5. **Lower resolution, bigger batch** — `IMG_SIZE=320`, `BATCH=32`, `EPOCHS=6`.
6. CLAHE intentionally **not** added — classmate tried it and it hurt their score.

**Unchanged:**
- EfficientNet-B0 + DenseNet121 (B3 swapped out earlier — too slow on MPS)
- 50/50 ensemble weight
- Multi-scale × hflip TTA at inference

---
## Step 1: Imports

In [ ]:
import copy
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.models import (
    DenseNet121_Weights,
    EfficientNet_B0_Weights,
    densenet121,
    efficientnet_b0,
)
from tqdm.auto import tqdm

print("Imports OK")

## Step 2: Reproducibility and device

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

## Step 3: Configuration

In [ ]:
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TRAIN_CSV = REPO / "data" / "train_val" / "train_val.csv"
TRAIN_IMG_DIR = REPO / "data" / "train_val" / "images"
TEST_IMG_DIR = REPO / "data" / "test_images"
PRED_DIR = REPO / "outputs" / "predictions"
CKPT_DIR = REPO / "outputs" / "checkpoints"
PRED_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 320                                      # low-res classmate-style setup
BATCH_SIZE = 16                                     # comfortable at 320 px on MPS
NUM_WORKERS = 0                                     # parallel PIL decode/augment — critical so the GPU isn't blocked
VAL_FRAC = 0.2

EPOCHS = 6                                          # matches classmate's EffNet epoch count
LR = 1e-4                                           # single aggressive LR — same for backbone + head

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# TTA: multi-scale × hflip. IMG_SIZE=320 → scales [320, 352, 384].
TTA_SCALES = [IMG_SIZE, IMG_SIZE + 32, IMG_SIZE + 64]

print(f"IMG_SIZE  : {IMG_SIZE}")
print(f"BATCH     : {BATCH_SIZE}")
print(f"WORKERS   : {NUM_WORKERS}")
print(f"EPOCHS    : {EPOCHS}")
print(f"LR        : {LR} (single — applied to both backbone and head)")
print(f"TTA scales: {TTA_SCALES} × {{orig, hflip}} = {2*len(TTA_SCALES)} passes per model")

## Step 4: Load labels

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df = df.rename(columns={
    "Image Index": "image_file",
    "Patient Age": "age",
    "Patient Sex": "sex",
    "Finding Labels": "finding",
})
df["label"] = (df["finding"] == "Cardiomegaly").astype(int)

print(df["finding"].value_counts())
print(f"\nPositive rate: {df['label'].mean():.3f}   Total: {len(df)}")

## Step 5: Dataset class

In [ ]:
class CardiomegalyDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, return_label=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.return_label = return_label

    def __len__(self):
        return len(self.df)

    def _load_image(self, fname):
        img = Image.open(self.img_dir / fname)
        if img.mode != "RGB":
            img = img.convert("RGB")
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load_image(row["image_file"])
        if self.transform is not None:
            img = self.transform(img)
        if not self.return_label:
            return img, row["image_file"]
        return img, torch.tensor(row["label"], dtype=torch.float32)

## Step 6: Stratified 80/20 split

In [ ]:
train_df, val_df = train_test_split(
    df, test_size=VAL_FRAC, stratify=df["label"], random_state=SEED
)
print(f"Train: {len(train_df)}  (pos rate {train_df['label'].mean():.3f})")
print(f"Val  : {len(val_df)}  (pos rate {val_df['label'].mean():.3f})")

# pos_weight for BCEWithLogitsLoss — corrects the slight positive-class under-representation
n_pos = int((train_df["label"] == 1).sum())
n_neg = int((train_df["label"] == 0).sum())
POS_WEIGHT = torch.tensor([n_neg / n_pos], dtype=torch.float32)
print(f"\nTrain class counts: pos={n_pos}  neg={n_neg}")
print(f"pos_weight = n_neg/n_pos = {POS_WEIGHT.item():.3f}")

## Step 7: Transforms and DataLoaders

Training uses the (mild) augmentation directly — no separate Stage 1 / Stage 2 loaders, because we're doing a single full-fine-tune stage.

In [ ]:
def build_transform(img_size, augment=False):
    if augment:
        # RandomResizedCrop fuses Resize + scale/translate jitter in one fast C op —
        # replaces the much slower PIL-based RandomAffine. Chest-X-ray-safe: no Solarize/Posterize/Equalize.
        ops = [
            T.RandomResizedCrop(img_size, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(brightness=0.2, contrast=0.2),
        ]
    else:
        ops = [T.Resize((img_size, img_size))]
    ops += [
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
    if augment:
        # RandomErasing operates on tensors — must come after ToTensor + Normalize
        ops.append(T.RandomErasing(p=0.25, scale=(0.02, 0.1), ratio=(0.3, 3.3)))
    return T.Compose(ops)

train_tf = build_transform(IMG_SIZE, augment=True)
val_tf = build_transform(IMG_SIZE, augment=False)

train_ds = CardiomegalyDataset(train_df, TRAIN_IMG_DIR, transform=train_tf)
val_ds = CardiomegalyDataset(val_df, TRAIN_IMG_DIR, transform=val_tf)

# persistent_workers keeps the worker processes alive between epochs — avoids re-spawn overhead
loader_kwargs = dict(num_workers=NUM_WORKERS)
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

print(f"Train batches: {len(train_loader)}   Val batches: {len(val_loader)}")

## Step 8: Peek at an augmented batch

In [ ]:
def denorm(x):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (x * std + mean).clamp(0, 1)

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, img, lbl in zip(axes.flat, imgs, labels):
    ax.imshow(denorm(img).permute(1, 2, 0).numpy())
    ax.set_title("Cardiomegaly" if lbl.item() else "No Finding", fontsize=9)
    ax.axis("off")
plt.suptitle(f"Augmented training batch @ {IMG_SIZE}px", fontsize=12)
plt.tight_layout()
plt.show()

## Step 9: Metrics helpers

In [ ]:
def sens_spec(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    pred = (y_prob >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    return sens, spec

def datathon_score(auroc, sens, spec):
    return 0.5 * auroc + 0.25 * sens + 0.25 * spec

def best_threshold_score(y_true, y_prob):
    """Threshold maximising datathon score via grid search."""
    auroc = roc_auc_score(y_true, y_prob)
    best_t, best_s = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 181):
        sens, spec = sens_spec(y_true, y_prob, threshold=t)
        s = datathon_score(auroc, sens, spec)
        if s > best_s:
            best_s, best_t = s, float(t)
    return best_t, best_s

## Step 10: Train / eval loops

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    ys, ps = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        logits = model(imgs).squeeze(1)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        ys.extend(labels.numpy().tolist())
        ps.extend(probs.tolist())
    ys, ps = np.array(ys), np.array(ps)
    auroc = roc_auc_score(ys, ps)
    sens, spec = sens_spec(ys, ps, threshold=0.5)
    return {"auroc": auroc, "sens": sens, "spec": spec, "y": ys, "p": ps}

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total = 0.0
    pbar = tqdm(loader, leave=False)
    for imgs, labels in pbar:
        imgs = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs).squeeze(1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total += loss.item()
        pbar.set_description(f"loss {loss.item():.3f}")
    return total / len(loader)

def train(model, tr_loader, vl_loader, optimizer, epochs, device, tag):
    # pos_weight compensates for class imbalance inside the loss
    criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT.to(device))
    best = {"auroc": -1.0, "state": None, "epoch": 0}
    history = []
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, tr_loader, optimizer, criterion, device)
        val = evaluate(model, vl_loader, device)
        dt = time.time() - t0
        score = datathon_score(val["auroc"], val["sens"], val["spec"])
        history.append({"epoch": epoch, "loss": tr_loss,
                        **{k: val[k] for k in ("auroc", "sens", "spec")}, "score": score})
        star = ""
        if val["auroc"] > best["auroc"]:
            best = {"auroc": val["auroc"], "state": copy.deepcopy(model.state_dict()), "epoch": epoch}
            star = " ★"
            torch.save(model.state_dict(), CKPT_DIR / f"{tag}_best.pt")
        print(
            f"[{tag}] ep {epoch:02d}/{epochs}  loss {tr_loss:.4f}  "
            f"AUROC {val['auroc']:.4f}  sens {val['sens']:.3f}  spec {val['spec']:.3f}  "
            f"score {score:.4f}  ({dt:.1f}s){star}"
        )
    model.load_state_dict(best["state"])
    return model, history, best

---
# Part 1 — EfficientNet-B0, full fine-tune

Every pretrained parameter is trainable. Differential LR: 3e-5 for the backbone, 1e-4 for the new binary head.

In [ ]:
def build_efnet_fullft():
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    # Nothing frozen — full fine-tune.
    in_features = model.classifier[1].in_features   # B0: 1280
    model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, 1))
    return model

efnet = build_efnet_fullft().to(device)
n_tr = sum(p.numel() for p in efnet.parameters() if p.requires_grad)
n_tot = sum(p.numel() for p in efnet.parameters())
print(f"EfficientNet-B0 full FT — trainable: {n_tr:,} / {n_tot:,} ({100*n_tr/n_tot:.2f}%)")

# Match classmate's recipe: plain Adam (not AdamW) with lr=1e-4, weight_decay=1e-4, single LR on all params.
opt_efnet = optim.Adam(efnet.parameters(), lr=LR, weight_decay=1e-4)
efnet, hist_ef, best_ef = train(
    efnet, train_loader, val_loader, opt_efnet,
    epochs=EPOCHS, device=device, tag="efnetb0_fullft"
)

---
# Part 2 — DenseNet121, full fine-tune

Same recipe — every pretrained layer unfrozen, differential LR.

In [ ]:
def build_dnet_fullft():
    model = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
    # Nothing frozen — full fine-tune.
    in_features = model.classifier.in_features      # DenseNet121: 1024
    model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, 1))
    return model

dnet = build_dnet_fullft().to(device)
n_tr = sum(p.numel() for p in dnet.parameters() if p.requires_grad)
n_tot = sum(p.numel() for p in dnet.parameters())
print(f"DenseNet121 full FT — trainable: {n_tr:,} / {n_tot:,} ({100*n_tr/n_tot:.2f}%)")

# Match classmate's recipe: plain Adam (not AdamW) with lr=1e-4, weight_decay=1e-4, single LR on all params.
opt_dnet = optim.Adam(dnet.parameters(), lr=LR, weight_decay=1e-4)
dnet, hist_dn, best_dn = train(
    dnet, train_loader, val_loader, opt_dnet,
    epochs=EPOCHS, device=device, tag="densenet121_fullft"
)

---
# Part 3 — Per-model validation evaluation

In [ ]:
ef_val = evaluate(efnet, val_loader, device)
dn_val = evaluate(dnet, val_loader, device)

rows = []
for name, v in [("efnetb0_fullft", ef_val), ("densenet121_fullft", dn_val)]:
    thr, s = best_threshold_score(v["y"], v["p"])
    sens_t, spec_t = sens_spec(v["y"], v["p"], threshold=thr)
    rows.append({"model": name, "auroc": v["auroc"], "thr": thr,
                 "sens": sens_t, "spec": spec_t, "score": s})

summary = pd.DataFrame(rows).sort_values("score", ascending=False)
print(summary.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, v, color in [
    ("EfficientNet-B0 fullFT", ef_val, "steelblue"),
    ("DenseNet121 fullFT",     dn_val, "orange"),
]:
    fpr, tpr, _ = roc_curve(v["y"], v["p"])
    ax.plot(fpr, tpr, lw=2, color=color, label=f"{name}  AUROC={v['auroc']:.3f}")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Validation ROC — full fine-tune per backbone (no TTA yet)")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
# Part 4 — TTA for each backbone

Multi-scale × horizontal-flip. `TTA_SCALES = [320, 352, 384]` × `{orig, hflip}` = 6 passes per model.

In [ ]:
@torch.no_grad()
def predict_probs_tta(model, loader, device, scales=TTA_SCALES, with_labels=True):
    """Average sigmoid probs across multi-scale × {orig, hflip} passes."""
    model.eval()
    all_probs, all_extra = [], []
    for imgs, extra in loader:                         # extra = labels (train/val) or filenames (test)
        imgs = imgs.to(device)
        acc = torch.zeros(imgs.size(0), device=device)
        passes = 0
        for scale in scales:
            x = imgs if scale == imgs.size(-1) else F.interpolate(
                imgs, size=(scale, scale), mode="bilinear", align_corners=False
            )
            acc += torch.sigmoid(model(x).squeeze(1))
            passes += 1
            acc += torch.sigmoid(model(torch.flip(x, dims=[3])).squeeze(1))
            passes += 1
        acc /= passes
        all_probs.extend(acc.cpu().numpy().tolist())
        if with_labels:
            all_extra.extend(extra.numpy().tolist())
        else:
            all_extra.extend(extra)
    return np.array(all_extra) if with_labels else all_extra, np.array(all_probs)

In [ ]:
print("TTA on EfficientNet-B0...")
y_val, ef_probs_tta = predict_probs_tta(efnet, val_loader, device)
ef_auc_tta = roc_auc_score(y_val, ef_probs_tta)
print(f"  single-pass AUROC: {ef_val['auroc']:.4f}")
print(f"  TTA AUROC       : {ef_auc_tta:.4f}  (Δ {ef_auc_tta - ef_val['auroc']:+.4f})")

print("\nTTA on DenseNet121...")
_, dn_probs_tta = predict_probs_tta(dnet, val_loader, device)
dn_auc_tta = roc_auc_score(y_val, dn_probs_tta)
print(f"  single-pass AUROC: {dn_val['auroc']:.4f}")
print(f"  TTA AUROC       : {dn_auc_tta:.4f}  (Δ {dn_auc_tta - dn_val['auroc']:+.4f})")

---
# Part 5 — 50/50 Ensemble

Plain mean of the TTA probabilities, matching the classmate's setup. Threshold re-tuned on the ensemble probabilities via grid search on the datathon score.

In [ ]:
ens_probs = 0.5 * ef_probs_tta + 0.5 * dn_probs_tta
ens_auc = roc_auc_score(y_val, ens_probs)
ens_thr, ens_score = best_threshold_score(y_val, ens_probs)
ens_sens, ens_spec = sens_spec(y_val, ens_probs, threshold=ens_thr)

print("=" * 55)
print("  Ensemble (50/50) validation results")
print("=" * 55)
print(f"  EfficientNet-B0 TTA AUROC : {ef_auc_tta:.4f}")
print(f"  DenseNet121     TTA AUROC : {dn_auc_tta:.4f}")
print(f"  Ensemble        TTA AUROC : {ens_auc:.4f}")
print("  " + "-" * 45)
print(f"  Best threshold            : {ens_thr:.3f}")
print(f"  Sensitivity               : {ens_sens:.3f}")
print(f"  Specificity               : {ens_spec:.3f}")
print(f"  Datathon score            : {ens_score:.4f}")
print("=" * 55)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, probs, color in [
    (f"EfficientNet-B0 TTA (AUROC={ef_auc_tta:.3f})", ef_probs_tta, "steelblue"),
    (f"DenseNet121 TTA   (AUROC={dn_auc_tta:.3f})",  dn_probs_tta, "orange"),
    (f"Ensemble 50/50    (AUROC={ens_auc:.3f})",      ens_probs,    "green"),
]:
    fpr, tpr, _ = roc_curve(y_val, probs)
    ax.plot(fpr, tpr, lw=2, color=color, label=name)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Validation ROC — TTA per backbone vs 50/50 ensemble")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
# Part 6 — Submission CSV

In [ ]:
test_files = sorted(p.name for p in TEST_IMG_DIR.iterdir() if p.suffix.lower() == ".png")
print(f"Test images: {len(test_files)}")

test_df = pd.DataFrame({"image_file": test_files, "label": 0})
test_tf = build_transform(IMG_SIZE, augment=False)
test_ds = CardiomegalyDataset(test_df, TEST_IMG_DIR, transform=test_tf, return_label=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("EfficientNet-B0 test-time TTA...")
names_ef, efnet_test_probs = predict_probs_tta(efnet, test_loader, device, with_labels=False)

print("DenseNet121 test-time TTA...")
names_dn, dnet_test_probs = predict_probs_tta(dnet, test_loader, device, with_labels=False)

assert list(names_ef) == list(names_dn), "Filename order mismatch between the two TTA passes"

test_ens_probs = 0.5 * efnet_test_probs + 0.5 * dnet_test_probs

sub = pd.DataFrame({
    "image_file": list(names_ef),
    "prob": test_ens_probs,
})
sub["pred"] = (sub["prob"] >= ens_thr).astype(int)
sub = sub.sort_values("image_file").reset_index(drop=True)

stamp = time.strftime("%Y%m%d_%H%M")
out_path = PRED_DIR / f"submission_efnetb0_densenet121_fullft_ensemble_TTA_{stamp}.csv"
sub.to_csv(out_path, index=False)

print(f"\nWrote {out_path}")
print(sub.head())
print(f"\nPositive rate in submission: {sub['pred'].mean():.3f}")

---
## Summary

**Changes from 05:** full fine-tune (no frozen layers), `IMG_SIZE=320`, `BATCH=32`, `EPOCHS=8`, differential LR (backbone 3e-5 / head 1e-4), single training stage. Still using EfficientNet-B0 + DenseNet121 (B3 was swapped back because it runs ~10× slower on MPS).

**Next experiments to try** (one at a time so the effect is readable):
1. Replace the mild augmentation block with `T.RandAugment(num_ops=2, magnitude=9)` or `T.TrivialAugmentWide()`.
2. Add CLAHE preprocessing inside `_load_image`.
3. Weighted ensemble (weight each backbone by its validation AUROC).
4. Retry EfficientNet-B3 on a CUDA machine if one is available.